In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2-T1 fixed delete/repeat local-path comparison

This is receiver-only: it reuses the named completed normal MP4 files, creates four second-lossy received videos, and performs 16 VAE re-encodes. It performs zero generation, transformer calls, or VAE decodes.


In [ ]:
from pathlib import Path
import sys, subprocess
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'c2a-2a-colab-preparation'
SOURCE = Path('/content/c2t1_local_paths_source')
if SOURCE.exists():
    existing_remote = subprocess.check_output(['git', '-C', str(SOURCE), 'remote', 'get-url', 'origin'], text=True).strip()
    if existing_remote != REPOSITORY_URL: raise RuntimeError(f'unexpected origin: {existing_remote}')
else:
    subprocess.run(['git', 'init', str(SOURCE)], check=True)
    subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_BRANCH], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', '--force', 'FETCH_HEAD'], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True)


In [ ]:
from datetime import datetime, timezone
CONFIG = SOURCE / 'runtime/c2t1/c2t1_local_path_run.json'
RUN_ID = 'c2t1_local_path_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/C2T1_LocalPath') / RUN_ID
print(CONFIG.read_text())
print('Output:', OUTPUT)
if OUTPUT.exists(): raise FileExistsError(str(OUTPUT))


In [ ]:
import os, signal
command = [sys.executable, '-u', '-m', 'runtime.c2t1.run_local_paths', '--config', str(CONFIG), '--output', str(OUTPUT)]
LOG = OUTPUT.parent / f'{RUN_ID}.launcher.log'
LOG.parent.mkdir(parents=True, exist_ok=True)
with LOG.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line); log.flush()
        returncode = process.wait()
    except BaseException:
        try: process.send_signal(signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
        raise
print('launcher exit', returncode)
print((OUTPUT / 'result.json').read_text() if (OUTPUT / 'result.json').exists() else 'No result file')
print('Drive log:', LOG)
if returncode: raise subprocess.CalledProcessError(returncode, command)


The run persists all four videos, 16 independent receiver observations, a no-local-path and a 97-path search for each video, compact candidate-to-equivalence-class JSONL mappings, class tables with scores/coverage, and the ideal difference check. A lower correct score alone is not evidence of gain because each wrong message receives the identical path search.
